# All reconstruction results, per dataset
For every dataset with results, one table puts side by side:
- **between frames** (`recon_eval.py`): the standard protocol, one 5-bin window between consecutive camera frames over the whole sequence;
- **fixed-duration windows** (`recon_eval_tms.py`): 5 bins of a fixed length (2, 5, 10, 20 ms, ...), with fixed-ms or EVREAL-style bins;
- optionally **EVREAL unchanged** (`recon_evreal_tms.py`), ground-truth scores only.

Two separate tables per dataset:
- **vs real**: each method's reconstruction compared with the reconstruction from the *real recorded events* of the same scene (the main result: how realistic the events are);
- **vs GT**: each reconstruction compared with the *camera frame*. The `real` row is the reference level; simulators were given these frames.

In every column the **top 3 methods per network are highlighted** (darkest = best, rank in brackets). `real` is never ranked. Values are the mean over sequences; ± is the std over sequences.

The newest finished run of each script is used per dataset; override with `RUNS` below. Run the scripts in a terminal first; this notebook only reads their reports.

In [ ]:
from pathlib import Path
import json, sys
import numpy as np
import pandas as pd
from IPython.display import display, HTML, Image as IPImage

REPO = Path.cwd().resolve()
while not (REPO / "benchmarks" / "recon_eval.py").is_file():
    REPO = REPO.parent
sys.path.insert(0, str(REPO / "benchmarks"))
import recon_eval

# ---- settings -------------------------------------------------------------
SPLIT = "test"
DATASETS = None            # None = every dataset with results, or e.g. ["eds", "hqevfi"]
RUNS = {}                  # optional per-dataset overrides, e.g. {"eds": {"recon": "reports/recon/eds/test/2026...",
                           #   "tms": "reports/recon_tms/eds/test/...", "evreal": None}}; None skips that script
OUR_METHOD = "evtween_v8_student"
TOP = 3                    # highlighted ranks per column
TABLE_METRICS = ("lpips", "ssim", "mse")   # metrics shown in the tables; e.g. ("lpips",) for narrower tables
SHOW_STD = True
INCLUDE_EVREAL = True
BEST_METRIC = "agreement_lpips"   # what "best reconstructed frame" means: agreement_* (vs real) or gt_* (vs GT)
BEST_NETWORK = None               # "e2vid", "firenet", or None for the mean of both
BEST_TOP = 1                      # frames shown per setting
BEST_SETTINGS = None              # None = every setting, or e.g. ["between frames", "10 ms bins, fixed"]

REAL, EVTWEEN, HIGHER = recon_eval.REAL, set(recon_eval.EVTWEEN), recon_eval.HIGHER_IS_BETTER
ORDER = {m: i for i, m in enumerate((REAL, *recon_eval.METHODS))}
METRICS = ("lpips", "ssim", "mse")
ARROW = {m: "↑" if HIGHER[m] else "↓" for m in METRICS}


def latest(base, marker):
    runs = sorted(p for p in base.glob("*") if (p / marker).is_file())
    return runs[-1] if runs else None


def runs_for(ds):
    own = RUNS.get(ds, {})

    def pick(key, kind, marker):
        if key in own:
            v = own[key]
            return None if v is None else (Path(v) if Path(v).is_absolute() else REPO / v)
        return latest(REPO / "reports" / kind / ds / SPLIT, marker)
    return dict(recon=pick("recon", "recon", "summary.csv"),
                tms=pick("tms", "recon_tms", "summary_all.csv"),
                evreal=pick("evreal", "recon_evreal", "evreal_summary.csv") if INCLUDE_EVREAL else None)


found = sorted({p.parent.name for kind in ("recon", "recon_tms", "recon_evreal")
                for p in (REPO / "reports" / kind).glob(f"*/{SPLIT}")})
datasets = DATASETS or found
RESULTS = {ds: runs_for(ds) for ds in datasets}
for ds, runs in RESULTS.items():
    print(ds)
    for k, v in runs.items():
        print(f"   {k:7s} {v if v else '-'}")

In [ ]:
def setting_label(bin_ms, bin_time, bins=5):
    return f"{bin_ms:g} ms bins, {'fixed' if bin_time == 'ms' else 'EVREAL-style'}"


def long_rows(summary, setting, key):
    rows = []
    for _, r in summary.iterrows():
        for kind in ("agreement", "gt"):
            for m in METRICS:
                c = f"{kind}_{m}"
                if f"{c}_mean" in summary and pd.notna(r[f"{c}_mean"]):
                    rows.append(dict(setting=setting, key=key, network=r["network"], method=r["method"], kind=kind,
                                     metric=m, mean=r[f"{c}_mean"], std=r.get(f"{c}_std", np.nan)))
    return rows


def load(runs):
    """All study results of one dataset as one long table (setting, network, method, kind, metric, mean, std)."""
    rows, notes = [], []
    if runs["recon"]:
        rows += long_rows(pd.read_csv(runs["recon"] / "summary.csv"), "between frames", (0, 0.0))
        notes.append("between frames: one 5-bin window between consecutive camera frames")
    if runs["tms"]:
        s = pd.read_csv(runs["tms"] / "summary_all.csv")
        cfg = json.loads((runs["tms"] / "config.json").read_text())
        bins = cfg.get("bins", 5)
        for _, g in s.groupby("config"):
            bms, bt = float(g["bin_ms"].iloc[0]), g["bin_time"].iloc[0]
            rows += long_rows(g, setting_label(bms, bt, bins), (1 if bt == "ms" else 2, bms))
        notes.append(f"N ms bins: back-to-back windows of {bins} x N ms; 'fixed' = fixed bin edges, "
                     "'EVREAL-style' = bins stretched between the window's first and last event")
    return pd.DataFrame(rows), notes


SHADES = {1: "#9fc3ef", 2: "#c7dbf6", 3: "#e3edfa"}   # one blue hue, darkest = best


def ranked(mean, higher_by_col, skip_rows=()):
    """Rank (1 = best) per column within each network; rows in skip_rows are not ranked."""
    ranks = pd.DataFrame(np.nan, index=mean.index, columns=mean.columns)
    for net in mean.index.get_level_values(0).unique():
        block = mean.loc[net]
        keep = [m for m in block.index if m not in skip_rows]
        for col in mean.columns:
            vals = block.loc[keep, col].dropna()
            r = vals.rank(ascending=not higher_by_col[col], method="min")
            for m, v in r.items():
                ranks.loc[(net, m), col] = v
    return ranks


def show_table(mean, std, higher_by_col, title, skip_rows=()):
    ranks = ranked(mean, higher_by_col, skip_rows)
    text = mean.copy().astype(object)
    css = pd.DataFrame("", index=mean.index, columns=mean.columns)
    for idx in mean.index:
        for col in mean.columns:
            v = mean.loc[idx, col]
            if pd.isna(v):
                text.loc[idx, col] = ""
                continue
            s = f"{v:.3f}"
            if SHOW_STD and std is not None and pd.notna(std.loc[idx, col]):
                s += f" ± {std.loc[idx, col]:.3f}"
            rk = ranks.loc[idx, col]
            if pd.notna(rk) and rk <= TOP:
                s += f" ({int(rk)})"
                css.loc[idx, col] = (f"background-color: {SHADES.get(int(rk), SHADES[3])}; color: #0b0b0b;"
                                     + (" font-weight: 700;" if rk == 1 else ""))
            elif idx[1] in skip_rows:
                css.loc[idx, col] = "color: #6b6a66; font-style: italic;"
            text.loc[idx, col] = s
    styler = text.style.apply(lambda _: css, axis=None).set_caption(title)

    def index_css(v):
        return "font-weight: 700; color: #1f5fa8;" if v == OUR_METHOD else ("font-weight: 600;" if v in EVTWEEN else "")
    try:
        styler = styler.map_index(index_css, axis=0, level=1)
    except AttributeError:  # pandas < 2.1
        styler = styler.applymap_index(index_css, axis=0, level=1)
    styler = styler.set_table_styles([
        {"selector": "caption", "props": "caption-side: top; font-size: 1.05em; font-weight: 600; text-align: left;"},
        {"selector": "th.col_heading", "props": "text-align: center;"},
        {"selector": "td", "props": "padding: 2px 8px; white-space: nowrap;"}])
    display(styler)


def study_tables(long, kind):
    d = long[long["kind"] == kind]
    if d.empty:
        return None
    settings = d.drop_duplicates("setting").sort_values("key")["setting"].tolist()
    cols = [(s, f"{m.upper()} {ARROW[m]}") for s in settings for m in TABLE_METRICS
            if not d[(d["setting"] == s) & (d["metric"] == m)].empty]
    d = d.assign(col=list(zip(d["setting"], d["metric"].map(lambda m: f"{m.upper()} {ARROW[m]}"))))
    mean = d.pivot_table(index=["network", "method"], columns="col", values="mean", aggfunc="first")
    std = d.pivot_table(index=["network", "method"], columns="col", values="std", aggfunc="first")
    mean = mean.reindex(columns=cols)
    std = std.reindex(columns=cols)
    mean.columns = std.columns = pd.MultiIndex.from_tuples(cols, names=["setting", "metric"])
    rows = sorted(mean.index, key=lambda i: (i[0], ORDER.get(i[1], 99)))
    mean, std = mean.loc[rows], std.loc[rows]
    higher = {c: c[1].endswith("↑") for c in mean.columns}
    return mean, std, higher

## Results per dataset

In [ ]:
LEGEND = ("Highlighted: top 3 per column within each network (darkest = 1st, bold = best, rank in brackets). "
          f"`real` is the reference and is not ranked. {OUR_METHOD} is in blue, other Evtween variants in bold.")
for ds, runs in RESULTS.items():
    display(HTML(f"<h2 style='margin-top:1.5em'>{ds}</h2>"))
    long, notes = load(runs)
    if long.empty:
        print("No recon_eval.py or recon_eval_tms.py results for this dataset")
    else:
        for n in notes:
            print("•", n)
        print(LEGEND)
        t = study_tables(long, "agreement")
        if t:
            show_table(*t, title=f"{ds}: vs real (agreement with the reconstruction from real events) - main result")
        t = study_tables(long, "gt")
        if t:
            show_table(*t, title=f"{ds}: vs GT (camera frame; real = reference level)", skip_rows=(REAL,))
    if runs["evreal"]:
        ev = pd.read_csv(runs["evreal"] / "evreal_summary.csv")
        ev = ev.assign(setting=ev["window_ms"].map(lambda w: "EVREAL between frames" if w == "between_frames"
                                                   else f"EVREAL {float(w):g} ms windows"),
                       key=ev["window_ms"].map(lambda w: -1.0 if w == "between_frames" else float(w)))
        mets = [m for m in TABLE_METRICS if f"{m}_frame_weighted" in ev]
        cols = [(s, f"{m.upper()} {ARROW[m]}") for s in ev.sort_values("key")["setting"].unique() for m in mets]
        mean = pd.DataFrame({c: ev[ev["setting"] == c[0]].set_index(["network", "method"])[
            f"{c[1].split()[0].lower()}_frame_weighted"] for c in cols})
        mean.columns = pd.MultiIndex.from_tuples(cols, names=["setting", "metric"])
        mean = mean.loc[sorted(mean.index, key=lambda i: (i[0], ORDER.get(i[1], 99)))]
        show_table(mean, None, {c: c[1].endswith("↑") for c in cols},
                   title=f"{ds}: EVREAL unchanged, vs GT only (frame-weighted; windows start at each stream's own first event)",
                   skip_rows=(REAL,))

## Best reconstructed frames for our method
For each dataset and setting: the saved frames (between frames) or windows (fixed duration) where `OUR_METHOD` scored best on `BEST_METRIC`, with their contact sheets (rows = networks; columns = ground truth, real, every method). Only saved frames have contact sheets (40 per sequence and setting), so this is the best among those.

In [ ]:
def best_frames(scores, key):
    """Rank saved frames/windows by OUR_METHOD's BEST_METRIC; also its rank among all methods on that frame."""
    metric = BEST_METRIC.split("_", 1)[1]
    s = scores[scores[BEST_METRIC].notna()]
    if BEST_NETWORK:
        s = s[s["network"] == BEST_NETWORK]
    table = s.groupby(["sequence", key, "method"])[BEST_METRIC].mean().unstack("method")
    if OUR_METHOD not in table:
        return None
    others = table.drop(columns=[REAL], errors="ignore")
    table["rank among methods"] = others.rank(axis=1, ascending=not HIGHER[metric], method="min")[OUR_METHOD]
    table["methods"] = others.notna().sum(axis=1)
    return table.sort_values(OUR_METHOD, ascending=not HIGHER[metric])[[OUR_METHOD, "rank among methods", "methods"]]


def show_best(table, sheets_dir, pattern, label):
    if table is None or table.empty:
        print(f"   {label}: {OUR_METHOD} has no saved frames here")
        return
    top = table.head(BEST_TOP)
    print(f"{label}")
    display(top.round(4))
    for (seq, idx), _ in top.iterrows():
        for path in sorted((sheets_dir / seq).glob(pattern(idx))):
            print("  ", seq, path.name)
            display(IPImage(filename=str(path)))


net = BEST_NETWORK or "mean of both networks"
for ds, runs in RESULTS.items():
    display(HTML(f"<h3>{ds}: best frames for {OUR_METHOD} by {BEST_METRIC} "
                 f"({'higher' if HIGHER[BEST_METRIC.split('_', 1)[1]] else 'lower'} is better, {net})</h3>"))
    shown = False
    if runs["recon"] and (BEST_SETTINGS is None or "between frames" in BEST_SETTINGS):
        sel = runs["recon"] / "selection.csv"
        if sel.is_file():
            saved = pd.read_csv(runs["recon"] / "per_frame.csv").merge(pd.read_csv(sel)[["sequence", "frame"]],
                                                                       on=["sequence", "frame"])
            show_best(best_frames(saved, "frame"), runs["recon"] / "contact_sheets", lambda k: f"{int(k):06d}_*.png",
                      "between frames")
            shown = True
        else:
            print("between frames: no saved frames; make them from the stored metrics with\n"
                  f"   python benchmarks/recon_eval.py {ds} --devices cuda:0 cuda:0 --run-dir {runs['recon']} --save hybrid")
    if runs["tms"] and (runs["tms"] / "selection.csv").is_file():
        sel_all = pd.read_csv(runs["tms"] / "selection.csv")
        summary = pd.read_csv(runs["tms"] / "summary_all.csv").drop_duplicates("config")
        for _, row in summary.sort_values(["bin_time", "bin_ms"], key=lambda c: c.map(lambda v: v != "ms")
                                          if c.name == "bin_time" else c).iterrows():
            label = setting_label(float(row["bin_ms"]), row["bin_time"])
            if BEST_SETTINGS is not None and label not in BEST_SETTINGS:
                continue
            cfg = row["config"]
            sel = sel_all[sel_all["config"] == cfg]
            if sel.empty or not (runs["tms"] / cfg / "per_window.csv").is_file():
                continue
            pw = pd.read_csv(runs["tms"] / cfg / "per_window.csv").merge(sel[["sequence", "window"]],
                                                                          on=["sequence", "window"])
            show_best(best_frames(pw, "window"), runs["tms"] / cfg / "contact_sheets", lambda s: f"w{int(s):06d}_*.png",
                      label)
            shown = True
    if not shown:
        print("No saved frames for this dataset")